# Notebook 05 — Sensitivity Analyses

**Purpose:** Robustness checks for the three main analyses: arm-level vs
paper-level baseline comparison, weighted vs unweighted pooling, and
influential observations via Cook's distance.

**Data Flow:**
| Direction | File | From / To |
|-----------|------|-----------|
| Input | `data/processed/arms.csv` | Notebook 00 |
| Input | `data/processed/papers.csv` | Notebook 00 |
| Output | `outputs/tables/table_S1–S4` | Thesis Discussion (Robustness) |

**Prerequisite notebooks:** N00 (arms + papers; all analyses run fresh).


## Section 0: Setup

In [1]:
# ── Setup ──────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

ROOT = Path().resolve().parent if Path().resolve().name == "notebooks"          else Path().resolve()
sys.path.insert(0, str(ROOT))

# ── External libraries ─────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
#   ^ OLS regression (R-style formulas)
from scipy import stats
#   ^ t-critical values and SEM for arm-level CIs

# ── Project modules ────────────────────────────────────────────────────────
# Data loading
from src.data_loading import load_arms
#   ^ Load + validate arm-level CSV

# Aggregation
from src.aggregation import compute_equity_scores
#   ^ Compute PROGRESS-Plus composite score (0–9) per paper

from src.aggregation import pool_across_papers, unweighted_mean_across_papers
#   ^ pool_across_papers() — weighted mean + 95% CI across papers (Cochran SE)
#   ^ unweighted_mean_across_papers() — simple mean across papers

from src.aggregation import aggregate_boolean_at_paper
#   ^ Collapse arm-level boolean → paper-level boolean (method="any_true")

# Statistics
from src.statistics import cooks_distance
#   ^ Cook's distance for each observation (flags > 4/n)

# ── Paths ──────────────────────────────────────────────────────────────────
OUTPUT_TABLES = ROOT / "outputs" / "tables"
OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)
(OUTPUT_TABLES / "pipeline").mkdir(parents=True, exist_ok=True)
ARMS_PATH = ROOT / "data" / "processed" / "arms.csv"
PAPERS_PATH = ROOT / "data" / "processed" / "papers.csv"

arms = load_arms(ARMS_PATH)
int_arms = arms[arms["arm"].isin(["treat1", "treat2"])]
papers = pd.read_csv(PAPERS_PATH)

print(f"Loaded: {len(papers)} papers, {len(arms)} arms")


Loaded: 64 papers, 134 arms


## Section 1: Arm-level vs paper-level baseline

In [2]:
# Arm-level vs paper-level baseline comparison. If arm sizes are balanced,
# these should agree. Large discrepancies suggest influential trials.
baseline_cols = ['age_mean', 'gender_pct_female', 'fev1_pct_mean', 'bmi_mean']
s2_rows = []

for col in baseline_cols:
    # Paper-level (already done in 02)
    p = pool_across_papers(papers, col, 'total_n')

    # Arm-level: each arm as a row, unweighted
    arm_vals = arms[col].dropna()
    arm_mean = float(arm_vals.mean())
    arm_se = float(stats.sem(arm_vals)) if len(arm_vals) > 1 else None
    t_crit = stats.t.ppf(0.975, df=len(arm_vals) - 1) if len(arm_vals) > 1 else None
    arm_ci_lo = arm_mean - t_crit * arm_se if t_crit else None
    arm_ci_hi = arm_mean + t_crit * arm_se if t_crit else None

    s2_rows.append({
        'variable': col,
        'paper_level_weighted': p['weighted_mean'],
        'paper_ci_lo': p['ci_lower'], 'paper_ci_hi': p['ci_upper'],
        'arm_level_simple': arm_mean,
        'arm_ci_lo': arm_ci_lo, 'arm_ci_hi': arm_ci_hi,
        'difference': abs(p['weighted_mean'] - arm_mean) if p['weighted_mean'] is not None else None,
        'n_papers': p['n_papers'], 'n_arms': len(arm_vals),
    })

s2 = pd.DataFrame(s2_rows)
s2.to_csv(OUTPUT_TABLES / 'pipeline' / 'table_S2_arm_vs_paper.csv', index=False)
print("Saved: table_S2_arm_vs_paper.csv")
print(s2.to_string())


Saved: table_S2_arm_vs_paper.csv
            variable  paper_level_weighted  paper_ci_lo  paper_ci_hi  arm_level_simple  arm_ci_lo  arm_ci_hi  difference  n_papers  n_arms
0           age_mean             69.787302    68.351062    71.223542         68.353556  67.769585  68.937528    1.433746        59     124
1  gender_pct_female             46.036645    41.031540    51.041750         40.510547  37.384896  43.636198    5.526098        61     128
2      fev1_pct_mean             47.510789    45.386008    49.635570         48.183206  46.397219  49.969193    0.672417        49     102
3           bmi_mean             26.462347    25.593509    27.331185         26.474783  25.917323  27.032242    0.012436        32      69


## Section 2: Weighted vs unweighted pooling

In [3]:
# Weighted vs unweighted pooling. Weighted: larger trials matter more
# (participant perspective). Unweighted: every trial = 1 vote.
# Difference reflects whether large trials enroll different populations.
s3_rows = []
for col in baseline_cols:
    p_w = pool_across_papers(papers, col, 'total_n')
    p_u = unweighted_mean_across_papers(papers, col)
    s3_rows.append({
        'variable': col,
        'weighted_mean': p_w['weighted_mean'],
        'weighted_ci_lo': p_w['ci_lower'], 'weighted_ci_hi': p_w['ci_upper'],
        'unweighted_mean': p_u['mean'],
        'unweighted_ci_lo': p_u['ci_lower'], 'unweighted_ci_hi': p_u['ci_upper'],
        'difference': abs(p_w['weighted_mean'] - p_u['mean'])
        if p_w['weighted_mean'] is not None and p_u['mean'] is not None else None,
        'n_papers': p_w['n_papers'],
    })

s3 = pd.DataFrame(s3_rows)
s3.to_csv(OUTPUT_TABLES / 'pipeline' / 'table_S3_weighted_vs_unweighted.csv', index=False)
print("Saved: table_S3_weighted_vs_unweighted.csv")
print(s3.to_string())


Saved: table_S3_weighted_vs_unweighted.csv
            variable  weighted_mean  weighted_ci_lo  weighted_ci_hi  unweighted_mean  unweighted_ci_lo  unweighted_ci_hi  difference  n_papers
0           age_mean      69.787302       68.351062       71.223542        68.398274         67.591858         69.204691    1.389028        59
1  gender_pct_female      46.036645       41.031540       51.041750        40.421383         36.019899         44.822867    5.615262        61
2      fev1_pct_mean      47.510789       45.386008       49.635570        48.161590         45.567759         50.755421    0.650801        49
3           bmi_mean      26.462347       25.593509       27.331185        26.533292         25.724266         27.342318    0.070945        32


## Section 3: Influential observations

In [4]:
# Influential observation detection via Cook's D > 4/n. Each model refit
# without flagged papers. Coefficients staying near zero + non-significant
# p-values → original null conclusion is robust.
# Scores recomputed here so N05 is self-contained.

# Need to rebuild scores as in notebook 03
papers['equity_score'] = compute_equity_scores(arms, papers)

ds_fields = [
    'digital_strategy_excludes', 'digital_strategy_provides_equipment',
    'digital_strategy_provides_training', 'digital_strategy_provides_ongoing_support',
]
ds_paper_vals = {}
for field in ds_fields:
    ds_paper_vals[field] = aggregate_boolean_at_paper(int_arms, field, method="any_true")
dis_df = pd.DataFrame({
    'ds_excludes': ds_paper_vals['digital_strategy_excludes'],
    'ds_equipment': ds_paper_vals['digital_strategy_provides_equipment'],
    'ds_training': ds_paper_vals['digital_strategy_provides_training'],
    'ds_support': ds_paper_vals['digital_strategy_provides_ongoing_support'],
})
usable_path = (~dis_df['ds_excludes'].astype(bool) | dis_df['ds_equipment'].astype(bool))
dis_df['digital_inclusiveness_score'] = (
    2 * (~dis_df['ds_excludes'].astype(bool)).astype(int)
    + dis_df['ds_equipment'].astype(int)
    + (dis_df['ds_training'].astype(bool) & usable_path).astype(int)
    + (dis_df['ds_support'].astype(bool) & usable_path).astype(int)
)
papers = papers.merge(dis_df[['digital_inclusiveness_score']], left_on='cov_nr', right_index=True, how='left')

# Prepare data subsets for each model
h1_data = papers[['cov_nr', 'equity_score', 'publication_year']].dropna()
h2a_data = papers[['cov_nr', 'age_mean', 'publication_year']].dropna()
h2b_data = papers[['cov_nr', 'fev1_pct_mean', 'publication_year']].dropna()
h3_data = papers[['cov_nr', 'digital_inclusiveness_score', 'equity_score', 'publication_year']].dropna()

models = {
    'H1': smf.ols('equity_score ~ publication_year', data=h1_data).fit(),
    'H2a': smf.ols('age_mean ~ publication_year', data=h2a_data).fit(),
    'H2b': smf.ols('fev1_pct_mean ~ publication_year', data=h2b_data).fit(),
    'H3': smf.ols('digital_inclusiveness_score ~ equity_score + publication_year', data=h3_data).fit(),
}

datasets = {'H1': h1_data, 'H2a': h2a_data, 'H2b': h2b_data, 'H3': h3_data}
outcomes = {'H1': 'equity_score', 'H2a': 'age_mean', 'H2b': 'fev1_pct_mean', 'H3': 'digital_inclusiveness_score'}
formulas = {
    'H1': 'equity_score ~ publication_year',
    'H2a': 'age_mean ~ publication_year',
    'H2b': 'fev1_pct_mean ~ publication_year',
    'H3': 'digital_inclusiveness_score ~ equity_score + publication_year',
}
beta_cols = {'H1': 'publication_year', 'H2a': 'publication_year', 'H2b': 'publication_year', 'H3': 'equity_score'}

s4_rows = []
for name, model in models.items():
    cd = cooks_distance(model)
    n = len(cd)
    threshold = 4 / n
    influential = np.where(cd > threshold)[0]
    print(f"{name}: {len(influential)} influential points (threshold={threshold:.4f})")

    if len(influential) > 0:
        data = datasets[name]
        influential_labels = data.index[np.where(cd > threshold)[0]]
        excluded_cov = data.loc[influential_labels, 'cov_nr'].tolist()
        print(f"  Excluding cov_nr: {excluded_cov}")

        refit = smf.ols(formulas[name], data=data.drop(index=influential_labels)).fit()
        orig_beta = model.params[beta_cols[name]]
        new_beta = refit.params[beta_cols[name]]
        refit_ci = refit.conf_int().loc[beta_cols[name]]
        refit_pval = refit.pvalues[beta_cols[name]]
        s4_rows.append({
            'model': name, 'n_original': n, 'n_influential': len(influential),
            'n_after': n - len(influential), 'original_beta': orig_beta,
            'refit_beta': new_beta, 'beta_change': new_beta - orig_beta,
            'refit_ci_lower': refit_ci[0], 'refit_ci_upper': refit_ci[1],
            'refit_p': refit_pval,
            'excluded_cov_nr': ','.join(map(str, excluded_cov)),
        })
    else:
        s4_rows.append({
            'model': name, 'n_original': n, 'n_influential': 0,
            'n_after': n, 'original_beta': model.params[beta_cols[name]],
            'refit_beta': model.params[beta_cols[name]], 'beta_change': 0,
            'refit_ci_lower': model.conf_int().loc[beta_cols[name], 0],
            'refit_ci_upper': model.conf_int().loc[beta_cols[name], 1],
            'refit_p': model.pvalues[beta_cols[name]],
            'excluded_cov_nr': '',
        })

s4 = pd.DataFrame(s4_rows)
s4.to_csv(OUTPUT_TABLES / 'pipeline' / 'table_S4_influential_obs_sensitivity.csv', index=False)
print("\nSaved: table_S4_influential_obs_sensitivity.csv")
print(s4.to_string())


H1: 6 influential points (threshold=0.0625)
  Excluding cov_nr: [4104, 4106, 4108, 4123, 4345, 4736]
H2a: 1 influential points (threshold=0.0678)
  Excluding cov_nr: [2125]
H2b: 1 influential points (threshold=0.0816)
  Excluding cov_nr: [2793]
H3: 5 influential points (threshold=0.0625)
  Excluding cov_nr: [770, 4106, 4123, 4345, 4736]

Saved: table_S4_influential_obs_sensitivity.csv
  model  n_original  n_influential  n_after  original_beta  refit_beta  beta_change  refit_ci_lower  refit_ci_upper   refit_p                excluded_cov_nr
0    H1          64              6       58      -0.054748    0.015510     0.070258       -0.082037        0.113056  0.751282  4104,4106,4108,4123,4345,4736
1   H2a          59              1       58      -0.125172   -0.114061     0.011111       -0.288683        0.060562  0.196053                           2125
2   H2b          49              1       48       0.172326    0.031242    -0.141084       -0.595624        0.658107  0.920528                

## Section 4: Results-ready statements

In [5]:
# Print results-ready English sentences for thesis cross-reference.
print("=== Primary results (re-fitted in N04) ===")

h1_beta = models['H1'].params['publication_year']
h1_ci = models['H1'].conf_int().loc['publication_year']
h1_p = models['H1'].pvalues['publication_year']
print(f"\nH1: Each additional year was associated with a {h1_beta:.3f} change in equity score "
      f"(95% CI [{h1_ci[0]:.3f}, {h1_ci[1]:.3f}], p = {h1_p:.4f}, n = {len(h1_data)}).")

h2a_beta = models['H2a'].params['publication_year']
h2a_ci = models['H2a'].conf_int().loc['publication_year']
h2a_p = models['H2a'].pvalues['publication_year']
print(f"\nH2a: Each additional year was associated with a {h2a_beta:.3f}-year change in mean age "
      f"(95% CI [{h2a_ci[0]:.3f}, {h2a_ci[1]:.3f}], p = {h2a_p:.4f}, n = {len(h2a_data)}).")

h2b_beta = models['H2b'].params['publication_year']
h2b_ci = models['H2b'].conf_int().loc['publication_year']
h2b_p = models['H2b'].pvalues['publication_year']
print(f"\nH2b: Each additional year was associated with a {h2b_beta:.3f} percentage-point change in FEV1% predicted "
      f"(95% CI [{h2b_ci[0]:.3f}, {h2b_ci[1]:.3f}], p = {h2b_p:.4f}, n = {len(h2b_data)}).")

h3_beta = models['H3'].params['equity_score']
h3_ci = models['H3'].conf_int().loc['equity_score']
h3_p = models['H3'].pvalues['equity_score']
print(f"\nH3: Each additional equity domain reported was associated with a {h3_beta:.3f} "
      f"change in digital inclusiveness score "
      f"(95% CI [{h3_ci[0]:.3f}, {h3_ci[1]:.3f}], p = {h3_p:.4f}, n = {len(h3_data)}), "
      f"adjusted for publication year.")

print("\n=== Influential obs sensitivity ===")
for _, row in s4.iterrows():
    direction = 'increased' if row['beta_change'] > 0 else 'decreased'
    print(f"\n{row['model']}: Excluding {int(row['n_influential'])} influential point(s) "
          f"{direction} the beta from {row['original_beta']:.3f} to {row['refit_beta']:.3f} "
          f"(change: {row['beta_change']:+.3f}).")
    if row['excluded_cov_nr']:
        print(f"  Excluded: {row['excluded_cov_nr']}")


=== Primary results (re-fitted in N04) ===

H1: Each additional year was associated with a -0.055 change in equity score (95% CI [-0.168, 0.059], p = 0.3393, n = 64).

H2a: Each additional year was associated with a -0.125-year change in mean age (95% CI [-0.326, 0.076], p = 0.2168, n = 59).

H2b: Each additional year was associated with a 0.172 percentage-point change in FEV1% predicted (95% CI [-0.444, 0.788], p = 0.5762, n = 49).

H3: Each additional equity domain reported was associated with a 0.047 change in digital inclusiveness score (95% CI [-0.200, 0.294], p = 0.7052, n = 64), adjusted for publication year.

=== Influential obs sensitivity ===

H1: Excluding 6 influential point(s) increased the beta from -0.055 to 0.016 (change: +0.070).
  Excluded: 4104,4106,4108,4123,4345,4736

H2a: Excluding 1 influential point(s) increased the beta from -0.125 to -0.114 (change: +0.011).
  Excluded: 2125

H2b: Excluding 1 influential point(s) decreased the beta from 0.172 to 0.031 (change:

## Section 6: Summary

In [6]:
# Summary cell.
print("=== Notebook 05 complete ===")
print(f"Completed at: {pd.Timestamp.now().isoformat()}")

=== Notebook 05 complete ===
Completed at: 2026-06-08T18:47:49.227233
